In [ ]:
###IMPORTS 
"""
Simple Gradient Inversion
Code inspired by https://github.com/daniel-scheliga/invertinggradients 18.05.2021
"""
import logging
from collections import OrderedDict
import datetime
import torch
from torch import nn
import torchvision
from torchvision import transforms
from scipy.optimize import linear_sum_assignment
import numpy as np
import matplotlib.pyplot as plt
from biomodule import BioModule
from bionet import datasets
import inversefed

In [ ]:
### EARLY STOPPING CLASS FOR FASTER RECONSTRUCTION
class EarlyStopping:
    """
    Early stops the training if validation loss doesn't improve after a given patience.
    Code inspired by https://github.com/Bjarten/early-stopping-pytorch 26.03.2021
    """
    def __init__(self, patience=7, delta=0, metric='loss', subject_to='min', verbose=False):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            metric (str): string of the metric we are looking at in our history
            subject_to (str): Defines whether the metric is subject to minimazation or maximization; 'min' or 'max' (defaut or when misspelled: 'min')
            verbose (bool): If True, logs a message for each validation loss improvement.
                            Default: False
        """
        self.patience = patience
        self.metric = metric
        self.subject_to = subject_to
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.stop = False
        self.improved = False
        self.delta = delta

    def get_state(self):
        return self.__dict__

    def set_state(self, state_dict):
        self.__dict__ = state_dict

    def __call__(self, metric):
        if self.subject_to == 'max':
            score = -metric
        else:
            score = metric

        if self.best_score is None:
            self.improved = True
            self.best_score = score
        elif score >= self.best_score + self.delta:
            self.improved = False
            self.counter += 1
            if self.verbose:
                logging.info(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.stop = True
        else:
            self.improved = True
            self.best_score = score
            self.counter = 0

In [ ]:
### MSE Metric Class
class Metric:
    def __init__(self):
        self.metric_fn = None
    def __call__(self, prediction, truth):
        if self.metric_fn == None:
            raise  NotImplementedError()
        else:
            return self.metric_fn(prediction, truth)
            
class MSE(Metric):
    def __init__(self, reduce=True):
        if reduce:
            self.metric_fn = torch.nn.MSELoss(size_average=None, reduction='mean')
        else:
            self.metric_fn = self.mse
        self.format = '.6f'
        self.reduce = reduce
        self.name = 'MSE'
        self.target = 'features'

    def mse(self, x, y):
        if not self.reduce:
            mse_fn = torch.nn.MSELoss(size_average=None, reduction='none')
            value = mse_fn(x, y)
            for _ in range(len(value.shape)-1):
                value = value.mean(dim=-1)
            return value
        else:
            print('You shouldn\'t be here.')

In [ ]:
### DEFINE SOME BASIC FUNCTIONS
def system_startup(args=None, defs=None, anomaly_detection=False, log_level=logging.INFO):
    """Set Logging"""
    rootLogger = logging.getLogger()
    logFormatter = logging.Formatter('%(asctime)s:[%(levelname)s][%(filename)s][%(funcName)s] %(message)s')
    consoleHandler = logging.StreamHandler(sys.stdout)
    consoleHandler.setFormatter(logFormatter)
    rootLogger.addHandler(consoleHandler)
    rootLogger.setLevel(log_level)

    """Log useful system information."""
    # Choose GPU device and print status information:
    torch.set_default_tensor_type(torch.cuda.FloatTensor)
    #torch.multiprocessing.set_start_method('spawn')
    device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    logging.info('Currently evaluating -------------------------------:')
    logging.info(datetime.datetime.now().strftime("%A, %d. %B %Y %I:%M%p"))
    logging.info(f'CPUs: {torch.get_num_threads()}, GPUs: {torch.cuda.device_count()}.')
    if args is not None:
        logging.info(args)
    if defs is not None:
        logging.info(repr(defs))
    if torch.cuda.is_available():
        logging.info(f'GPU : {torch.cuda.get_device_name(device=device)}')
    if anomaly_detection:
        torch.autograd.set_detect_anomaly(True)
        logging.info('Activating torch.autograd anomaly detection')
    return device
    
def build_mnist(batch_size = 64, datapath='../../bionet/data', train_transformations = transforms.ToTensor(), val_transformations = transforms.ToTensor()):
    trn_set = datasets.Datasets(dataset_name = 'MNIST', mode='train', path=datapath, cv=0, cl_filter=None, verbose=False)
    #trn_set = torchvision.datasets.CIFAR10(root=datapath, train=True, download=False, transform=train_transformations)
    trn_loader = torch.utils.data.DataLoader(trn_set, batch_size=batch_size, shuffle=False, drop_last=True)
    return trn_loader
    
def build_cifar100(batch_size = 64, datapath='../../bionet/data', train_transformations = transforms.ToTensor(), val_transformations = transforms.ToTensor()):
    trn_set = datasets.Datasets(dataset_name = 'CIFAR100', mode='train',path=datapath, cv=0, cl_filter=None, verbose=False)
    #trn_set = torchvision.datasets.CIFAR10(root=datapath, train=True, download=False, transform=train_transformations)
    trn_loader = torch.utils.data.DataLoader(trn_set, batch_size=batch_size, shuffle=False, drop_last=True)
    return trn_loader

def build_cifar10(batch_size = 64, datapath='../../bionet/data', train_transformations = transforms.ToTensor(), val_transformations = transforms.ToTensor()):
    trn_set = datasets.Datasets(dataset_name = 'CIFAR10', mode='train',path=datapath, cv=0, cl_filter=None, verbose=False)
    #trn_set = torchvision.datasets.CIFAR10(root=datapath, train=True, download=False, transform=train_transformations)
    trn_loader = torch.utils.data.DataLoader(trn_set, batch_size=batch_size, shuffle=False, drop_last=True)
    return trn_loader

def get_gradient(model, input_data, gt_labels, loss_fn, train_mode, device, verbose=False):
    if verbose: print('Generatig gradient from victim data...')
    if train_mode:
        model.train()
    else:
        model.eval()

    model.zero_grad()
    target_loss = loss_fn(model(input_data), gt_labels)
    
    #parameters = [params for params in model.parameters() if params.requires_grad ]
    
    gradient = torch.autograd.grad(target_loss, model.parameters(), allow_unused=True)
    #gradient = torch.autograd.grad(target_loss, parameters, allow_unused=True)
    
    #return gradient
    gradient = [grad.detach() for grad in gradient if type(grad) == torch.Tensor]
    #for g in gradient:
    #    print(g)
    return gradient

def get_gradient_2(model, input_data, gt_labels, loss_fn, train_mode, device, verbose=False):
    if verbose: print('Generatig gradient from victim data...')
    if train_mode:
        model.train()
    else:
        model.eval()

    model.zero_grad()
    target_loss = loss_fn(model(input_data), gt_labels)
    gradient = torch.autograd.grad(target_loss, model.parameters(), allow_unused=True)
    return gradient

def gradient_inversion(gradient, labels, model, data_shape, dm, ds, batch_size, device, verbose=False):
    if verbose: print('Performing a Gradientinversion attack.')
    #build inversefed library specific config for the reconstruction attack
    c = dict(signed=True,
              boxed=True,
              cost_fn='sim',
              indices='def',
              weights='equal',
              lr=0.1,
              optim='adam',
              restarts=1,
              max_iterations=7000,
              total_variation=1e-6,
              init='randn',
              filter='none',
              lr_decay=True,
              scoring_choice='loss',
              loss_fn = torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean'), #(Loss fn the model was trianed with)
              early_stopper = EarlyStopping(1000, 0, 'ReconstructionLoss', 'min', False)
              )
    rec_machine = inversefed.GradientReconstructor(model, (dm, ds), c, num_images=batch_size)
    output, stats = rec_machine.reconstruct(gradient, labels, img_shape=data_shape)

    return output, stats['opt']


def match_reconstructions(images, reconstructions):
    cost_matrix = get_similarity_cost_matrix(images, reconstructions)
    rec_idx = linear_sum_assignment(cost_matrix, maximize=True)[1]
    return reconstructions[rec_idx].detach().clone()


def get_similarity_cost_matrix(images, reconstructions):
    m = MSE(False)
    cost_matrix = []
    for img in images:
        i, r = torch.broadcast_tensors(img, reconstructions)
        current_metric = m(i, r)
        cost_matrix.append(np.array(current_metric).astype(float))
    return np.array(cost_matrix)

def show_single_img(img):
    plt.imshow(img.permute(1, 2, 0).cpu());
    plt.axis('off')
    plt.show()
    plt.close()
DEVICE = system_startup(log_level=logging.ERROR)

In [ ]:
def test_net(model, loss_fn, batch_size=8, no_batches=2, dataset='CIFAR100', verbose=False):
    losses, outputs = [],[]
    if dataset != 'CIFAR100': 
        raise NotImplementedError
    trn_loader = build_cifar100(batch_size=batch_size)
    trn_loader = iter(trn_loader)
    print('batch no: ', end='')
    for a in range(no_batches):      
        print(f'... {a} ', end='')
        #Prep dataset        
        images, labels = next(trn_loader)
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        batch_size = images.shape[0]
        
        data_shape = images[0].shape
        cifar100_mean = [0.4914672374725342, 0.4822617471218109, 0.4467701315879822]
        cifar100_std = [0.24703224003314972, 0.24348513782024384, 0.26158785820007324]
        dm = torch.as_tensor(cifar100_mean)[:, None, None]
        ds = torch.as_tensor(cifar100_std)[:, None, None]

        gradient = get_gradient_2(model, images, labels, loss_fn, True, DEVICE)
        output, best_loss = gradient_inversion(gradient, labels, model, data_shape, dm, ds, batch_size, DEVICE)
        losses.append(best_loss)
        outputs.append(output)
        
    if verbose:
        images = images.detach().clone().cpu()
        output = output.detach().clone().cpu()
        for (i,l), r in zip(zip(images,labels), output):
            logging.info(f"Image Label: {l.item()}")
            show_single_img(i)
            show_single_img(r)
    return np.array(best_loss).mean(),{'losses':losses, 'outputs':outputs}
        

In [ ]:
import os
import pickle as pkl
from bionet.modules.linear import FCNet
from bionet.modules.alexnetMini import AlexNetMini
from bionet.modules.resnetMini import resnet20,resnet32,resnet44,resnet56,resnet110,resnet1202

def hms_from_seconds(time_el):
    h_el = int(time_el//3600)
    m_el = int((time_el-3600*h_el)//60)
    s_el = int(time_el-3600*h_el-m_el*60)
    return h_el, m_el, s_el

def eval_resnet(model_name, num_channels, num_classes):
    if '1202' in model_name:
        return resnet1202(num_channels, num_classes)
    elif '32' in model_name:
        return resnet32(num_channels, num_classes)
    elif '44' in model_name:
        return resnet44(num_channels, num_classes)
    elif '56' in model_name:
        return resnet56(num_channels, num_classes)
    elif '110' in model_name:
        return resnet110(num_channels, num_classes)
    else :
        return resnet20(num_channels, num_classes)

filter_accums = 0

path = 'models'
dirlist = os.listdir(path)

batch_sizes = [2,4,8]
num_batch_runs = 5

# mint run
MINT = True

results_list = []
from time import time
t0 = time()
num_runs = len(dirlist)
i=0
for batch_size in  batch_sizes:
    for d in dirlist:    
        print("Working with: ",d)

        result = pkl.load(open(path+d,'rb'))
        if filter_accums == result['result']['options']['accum_neurons']:
            converter = BioModule.get_convert_to_bionet_converter(accum_neurons=result['result']['options']['accum_neurons'])
        else:
            continue
        print(result['result']['model'])
        if result['result']['model'] == 'FCNet':
            if MINT: 
                model = converter(FCNet)(**result['result']['options']['model_args'])
            else:
                model = FCNet(**args)
        elif result['result']['model'] == 'AlexNetMini':
            if MINT: 
                model = converter(AlexNetMini)(**result['result']['options']['model_args'])
            else:
                model=AlexNetMini(**result['result']['options']['model_args'])
        elif 'resnet' in result['result']['model']:
            model_cl, args = eval_resnet(result['result']['model'], 3, 100)
            if MINT: 
                model = converter(model_cl)(**args)
            else:
                model = model_cl(**args)


        for model_dict in result['result']['models']:
            print(f"Evaluating {len(model_dict)} models")
            t1 = time()
            time_el = int(t1-t0)
            h_el, m_el, s_el = hms_from_seconds(time_el)
            approx_time_to_go = 0 if i==0 else (time_el/i)*num_runs
            h_to, m_to, s_to = hms_from_seconds(approx_time_to_go)
            epoch = model_dict['epoch']      
            model.load_state_dict(model_dict['state_dict'])    

            criterion =  torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean')
            skip=False
            for saved_result_row in results_list:
                if result['result']['model'] == saved_result_row['model_cl'] and\
                    result['result']['options']['purge'] == saved_result_row['options']['purge'] and\
                    result['result']['options']['scale_grad'] == saved_result_row['options']['scale_grad'] and\
                    result['result']['options']['crystallize'] == saved_result_row['options']['crystallize'] and\
                    result['result']['options']['accum_neurons'] == saved_result_row['options']['accum_neurons'] and\
                    model_dict['epoch'] == saved_result_row['epoch'] and\
                    batch_size == saved_result_row['batch_size']:                        
                        print("SKIPPING")
                        print(result['result']['model'],result['result']['options']['purge'],result['result']['options']['scale_grad'],result['result']['options']['crystallize'],result['result']['options']['accum_neurons'], model_dict['epoch'] ,"time el", time_el, "to go", f"{h_to}:{m_to}:{s_to}")
                        skip=True
                        break
            if skip: continue
            print("Working on: ",result['result']['model'],result['result']['options']['purge'],result['result']['options']['scale_grad'],result['result']['options']['crystallize'],result['result']['options']['accum_neurons'], model_dict['epoch'] ,"time el", time_el, "to go", f"{h_to}:{m_to}:{s_to}")                
            if 'CIFAR100' in d: dataset='CIFAR100'
            model.cuda()
            loss, results = test_net(model, criterion, batch_size,num_batch_runs, dataset, False)      
            print(loss)

            results_list.append({'loss':loss,'model_cl': result['result']['model'], 'epoch':epoch, 'options':result['result']['options'],'results':results,'batch_size':batch_size})
            i+=1
            